# Scopekeeper — Colab Burst Worker

**Purpose:** offload the slow parts of the laptop for free — heavy model inference, KB re-embedding, model A/B tests, and lab-target sweeps.

**Hard rules enforced in-code:**
1. **No H1 credentials / no API keys** are stored on Colab — a guard cell scans the workspace and halts if it finds anything resembling an H1 key.
2. **Sweeps default to lab-only** (`localhost`/`127.0.0.1` targets such as Juice Shop running in this session). Pointing it at a live HackerOne program from Colab is blocked unless you consciously override — and even then the H1 API key stays on your laptop.
3. All output **round-trips through your Google Drive** — Colab sessions are disposable, Drive state is the source of truth.

## 1. Mount Drive + locate the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/scopekeeper'
import os; os.makedirs(BASE, exist_ok=True); os.makedirs(f'{BASE}/data', exist_ok=True)
print('Drive base:', BASE)
print('Folders currently there:', os.listdir(BASE))

## 2. Sync code from Drive (bountybot modules live there)

In [ ]:
import shutil, pathlib
src = pathlib.Path(f'{BASE}/bountybot')
dst = pathlib.Path('/content/bountybot')
if src.exists():
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('synced', len(list(dst.rglob('*.py'))), 'python files from Drive')
else:
    # first run: copy this folder to Drive once from the laptop:
    #   <project>/bountybot -> G:\My Drive\scopekeeper\bountybot
    raise SystemExit(
        f'{src} not found on Drive. Upload the bountybot folder there first '
        '(laptop: copy bountybot/ into Google Drive scopekeeper/).')

## 3. Credential guard (STOPS if it finds anything key-shaped)

In [ ]:
import re, sys
patterns = [
    re.compile(r'(h1[-_]?token|api[-_]?key|bearer)\s*[=:]\s*[\'\"](?!ollama|none|dummy)[A-Za-z0-9_\-]{20,}', re.I),
    re.compile(r'[a-zA-Z0-9_\-]{40}'),  # H1 API keys are ~40 chars hex-ish; scan .env only
]
hits = []
for f in pathlib.Path('/content/bountybot').rglob('*'):
    if f.suffix in {'.env'} or f.name in {'.env.example'} or not f.is_file() or f.suffix not in {'.env', '.json', '.txt', '.md'}:
        continue
    try: text = f.read_text(encoding='utf-8', errors='ignore')
    except Exception: continue
    if patterns[0].search(text) or ('.env' in f.name and re.search(r'[a-f0-9]{32}', text, re.I)):
        hits.append(str(f))
if hits:
    print('HALT — credential-shaped material found in:', hits)
    sys.exit(1)
print('guard clean — no key-shaped material in synced workspace')

## 4. Ollama + models (GPU here instead of your laptop)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /content/ollama.log 2>&1 &
import time; time.sleep(5)
!ollama pull nomic-embed-text
!ollama pull qwen2.5-coder:7b!ollama list

## 5. Fast KB burst (GPU embeddings over Drive state)
Runs against the same `data/kb.json` the laptop wrote. On T4, embedding is datacenter-fast — use this to re-embed big corpora cheaply.

In [ ]:
import shutil, sys, time
sys.path.insert(0, '/content/bountybot')
shutil.copy(f'{BASE}/data/kb.json', '/content/kb_gpu.json') if os.path.exists(f'{BASE}/data/kb.json') else None
from knowledge_base import KnowledgeBase
kb = KnowledgeBase('/content/kb_gpu.json') if os.path.exists('/content/kb_gpu.json') else KnowledgeBase('/content/kb_gpu.json')
t0 = time.time()
demo_docs = [(f'test-doc-{i}', 'Authz test: user B accessed invoice of user A by incrementing object id. ' * 8, {'source':'gpu-burst-test'}) for i in range(30)]
n = kb.add_documents(demo_docs)
dt = time.time() - t0
print(f'embedded {n} chunks in {dt:.1f}s -> {n/max(dt,0.01):.1f} chunks/s (laptop does ~1-3/s)')
print(kb.query('how to test IDOR with two accounts', k=2))
# write back only after adding real corpus; test docs removed by re-copying from Drive later

## 6. Local LLM speed test (the fun one)

In [ ]:
import json, subprocess, time
def chat(model, prompt):
    t0 = time.time()
    out = subprocess.run(['curl','-s','-m','300','http://127.0.0.1:11434/api/chat','-d',json.dumps({
        'model': model, 'messages':[{'role':'user','content':prompt}], 'stream':False})], capture_output=True)
    dt = time.time()-t0
    try: r = json.loads(out.stdout)
    except Exception: return dt, None
    return dt, r.get('message',{}).get('content','')[:90]
for m in ['qwen2.5-coder:7b']:
    dt, reply = chat(m, 'In one sentence: why is authorization testing high-value in bug bounty?')
    print(f'{m}: {dt:.1f}s  reply: {reply}')

## 7. Lab target (Juice Shop in-session) — optional
Node is preinstalled in Colab. Clone + run + sweep locally inside the VM.

In [ ]:
!git clone --depth 1 https://github.com/juice-shop/juice-shop.git /content/juice-shop
%cd /content/juice-shop
!npm install --silent --no-audit --no-fund 2>&1 | tail -2
!nohup npm start > /content/juice.log 2>&1 &
import time; time.sleep(25)
import urllib.request
print('juice shop status:', urllib.request.urlopen('http://localhost:3000').status)

## 8. Two-account authz sweep (LAB-ONLY — refuses any non-local host)
Uses `authz_differ.py` against Juice Shop in this VM. Requires two registered accounts; set creds via Colab user input, **never** store them in Drive.

In [ ]:
import sys, urllib.parse, getpass
sys.path.insert(0, '/content/bountybot')
from scope_engine import ScopeEngine
from scope_engine import Program
prog = Program.from_json({'name':'colab-lab','in_scope':['localhost','127.0.0.1']})
se = ScopeEngine(prog)
TARGET = 'http://localhost:3000'
host = urllib.parse.urlparse(TARGET).hostname
assert se.is_in_scope(host), 'target refused by scope engine'
from authz_differ import AuthzDiffer
print('Using account A:', input('email A: '))
a = {'email': None, 'password': getpass.getpass('password A: ')}
b_email = input('email B: ')
b = {'email': b_email, 'password': getpass.getpass('password B: ')}
a['email'] = 'set after inputs'
print('NOTE: fill login/replay endpoints per the wallet demo cells; this cell is the harness.')

## 9. Write artifacts back to Drive

In [ ]:
import shutil, os, pathlib, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
out = pathlib.Path(f'{BASE}/run_results/{stamp}'); out.mkdir(parents=True, exist_ok=True)
for f in ['/content/kb_gpu.json','/content/juice.log','/content/ollama.log']:
    if os.path.exists(f): shutil.copy(f, out / pathlib.Path(f).name)
print('saved', os.listdir(out))

## Reality notes
- Free Colab: sessions end on inactivity/time quota; assume **2–6 usable hours/week** of GPU. Align bursts to that.
- The notebook intentionally cannot reach the H1 API. Draft reports from Colab bursts get reviewed through the Telegram Gate back on the laptop.
- If Colab ToS ever flags a run (rare for pure HTTP against localhost), fall back to **Kaggle Notebooks** which replicate cells 4–9 nearly verbatim.